In [28]:
import pandas as pd
import json
import seaborn as sns
import numpy as np
import networkx as nx
from pyvis.network import Network

import sys
sys.path.append("../functions/")
import networks

import network_diff_ev

In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = media_db[media_db.medium=="M9"].compound.values
compounds = list(compounds)
compounds.remove("glc__D")
m9_compounds = ["M_"+met+"__91__e__93__" for met in compounds]

with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

results = pd.read_csv("../output/Venturelli2018/results_steadiercom/NONE_abundance.tsv",sep="\t")
results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]
results = results[~results.compound.isin(m9_compounds)]
results_env = results[(results.donor=="environment")].copy()
results_env.receiver = results_env.receiver.map(lambda x:x.split("_")[0][0].upper()+x.split("_")[1][0].upper())

dataset4 = pd.read_excel("../data/Venturelli2018/msb178157-sup-0005-datasetev4.xlsx",sheet_name="Dataset EV4",index_col=0)
dataset4.columns = dataset4.columns.map(lambda x: x.replace("'",""))
dataset4.columns = dataset4.columns.map(lambda x: x.replace(" ",""))

### Differential evolution to find best values for: 


SteadierCom results filtering:
- freq_cutoff
- mass_cutoff
- cutoff_overlap

Experimental results filtering:
- exp_cutoff1: Limit on log change of concentration
- exp_cutoff2: Limit on number of connections based on number of compounds


In [57]:
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from scipy.optimize import differential_evolution


# Initialize any shared variables if needed
pop_min = []
energies_min = []

def maplike_fun(func, x, *args, max_workers=2):
    # Store the current population
    pop_min.append(np.copy(x))

    # Evaluate the objective function for each individual in parallel
    e = []
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit each individual to the executor, running them in parallel
        futures = [executor.submit(func, x_i, *args) for x_i in x]
        
        # Collect the results as they complete
        for future in futures:
            e.append(future.result())
    
    # Store the energies (objective function values)
    energies_min.append(e)
    print(e)
    
    
    # Return the objective values, ensuring it matches the length of x
    return e



In [60]:
pop_min = []
energies_min = []
res_min = differential_evolution(network_diff_ev.inner_problem,
                  bounds=[(-6,0),(-6,0),(0,10),(-10,0),(0,10)],
                  args=(results_env,dataset4),
                  workers=maplike_fun,
                  polish=False,
                  seed=1,
                  updating='deferred')

[inf, inf, inf, inf, inf, inf, inf, np.float64(0.7381825309556787), np.float64(0.924044547468725), inf, inf, inf, np.float64(0.7113248654051871), inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, np.float64(0.7484423525312737), inf, np.float64(0.8855682336236558), np.float64(0.6696528555431132), np.float64(0.944529980377477), inf, np.float64(1.0219264504826757), inf, inf, inf, np.float64(0.750576702036838), inf, inf, inf, inf, inf, inf, inf, np.float64(0.6012305252190258), inf, inf, inf, inf, inf, np.float64(0.42728715746894586), inf, inf, inf, inf, inf, inf, inf, np.float64(0.961424162509477), inf, inf, inf, inf, inf, inf]
[np.float64(0.944529980377477), np.float64(0.9211373814644015), inf, inf, inf, np.float64(0.5627626839023969), np.float64(0.7373871342805549), np.float64(0.713980916170069), np.float64(0.944529980377477), inf, inf, inf, np.float64(0.639444872453601), inf, inf, inf, np.float64(0.8444572457904362), np.fl

In [91]:
res_min

             message: Optimization terminated successfully.
             success: True
                 fun: 0.09630388588493599
                   x: [-1.759e-01 -5.262e+00  2.526e+00 -1.828e+00
                        2.343e-01]
                 nit: 48
                nfev: 3675
          population: [[-1.759e-01 -5.262e+00 ... -1.828e+00  2.343e-01]
                       [-1.799e-01 -5.275e+00 ... -1.768e+00  1.854e-01]
                       ...
                       [-1.888e-01 -5.245e+00 ... -1.765e+00  3.806e-01]
                       [-2.108e-01 -5.265e+00 ... -1.800e+00  2.447e-01]]
 population_energies: [ 9.630e-02  9.630e-02 ...  9.630e-02  9.630e-02]

In [92]:
res_min.x

array([-0.17591952, -5.26166681,  2.52588296, -1.82796114,  0.23428625])

### Show solution

In [84]:
import matplotlib as mpl
# Color end points for colorFader
b1='#f8cdcf' #bright pink
b2='#fa4f56' #red

c1='#ffffff' #white
c2='#0b5394' #blue
    
## From https://stackoverflow.com/questions/25668828/how-to-create-colour-gradient-in-python
def colorFader(c1,c2,mix=0): #fade (linear interpolate) from color c1 (at mix=0) to c2 (mix=1)
    c1=np.array(mpl.colors.to_rgb(c1))
    c2=np.array(mpl.colors.to_rgb(c2))
    return mpl.colors.to_hex((1-mix)*c1 + mix*c2)





def competition_network(network_data,network_exp,closeness_centrality_met,cutoff=0):
    net = Network()
    
    maximum_centrality = pd.Series(closeness_centrality_met).max()

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    
    
    if isinstance(closeness_centrality_met,pd.Series):
        value = [(closeness_centrality_met[node]*10) for node in nodes]
        color = [colorFader(b1,b2,closeness_centrality_met[node]/maximum_centrality) for node in nodes]
        net.add_nodes(nodes,color=color,value=value)
    else:
        net.add_nodes(nodes)
    maximum_overlap = network_data.max()
    
    i=0
    j=0
    
    for index,value in network_data.items():
        #new_index1 = index[0].split("_")[0][0].upper()+index[0].split("_")[1][0].upper()
        #new_index2 = index[1].split("_")[0][0].upper()+index[1].split("_")[1][0].upper()
        
        new_index = index#(new_index1,new_index2)
        new_index_rev = (index[1],index[0]) #(new_index2,new_index1)



        if value>cutoff:
            if (new_index in network_exp.index) or (new_index_rev in network_exp.index):
                color = "red"
                i+=1
            else:
                color = colorFader(c1,c2,value/maximum_overlap)
            net.add_edge(index[0],index[1],width=value*2,color=color)
            j+=1
        
    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)

    print(f"Matching exp for {(i/j)*100}% of interactions")
    
    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net


In [85]:

# Get values...
combination =res_min.x

freq_cutoff,mass_cutoff,cutoff_overlap,exp_cutoff1,exp_cutoff2 = combination
freq_cutoff = 10**freq_cutoff
mass_cutoff = 10**mass_cutoff

# Get data for metabolites..
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = media_db[media_db.medium=="M9"].compound.values
compounds = list(compounds)
compounds.remove("glc__D")
m9_compounds = ["M_"+met+"__91__e__93__" for met in compounds]

with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

# Read steadiercom results and process..
results = pd.read_csv("../output/Venturelli2018/results_steadiercom/NONE_abundance.tsv",sep="\t")
results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]
results = results[~results.compound.isin(m9_compounds)]
results_env = results[(results.donor=="environment")].copy()
results_env.receiver = results_env.receiver.map(lambda x:x.split("_")[0][0].upper()+x.split("_")[1][0].upper())

# Read experimental results and process
dataset4 = pd.read_excel("../data/Venturelli2018/msb178157-sup-0005-datasetev4.xlsx",sheet_name="Dataset EV4",index_col=0)
dataset4.columns = dataset4.columns.map(lambda x: x.replace("'",""))
dataset4.columns = dataset4.columns.map(lambda x: x.replace(" ",""))

# Find consumed metabolites - experiments
consumption = dataset4<=exp_cutoff1
consumption = consumption.map(int)
network_exp = consumption.T.dot(consumption)
network_exp = network_exp.mask(np.tril(np.ones(network_exp.shape)).astype(np.bool)).stack() 
network_exp = network_exp[network_exp>exp_cutoff2]

# Cutoff mass_rate*frequency
results_env = results_env[results_env["mass_rate*frequency"]>mass_cutoff].copy()

# Cutoff frequency and generate network
network_data = networks.competition_network_data(results_env,freq_cutoff=freq_cutoff)


In [89]:
# exp_cutoff1: Limit on log change of concentration
# exp_cutoff2: Limit on number of connections based on number of compounds

[freq_cutoff,mass_cutoff,cutoff_overlap,exp_cutoff1,exp_cutoff2]

[np.float64(0.6669303496620699),
 np.float64(5.474357965125416e-06),
 np.float64(2.52588296390405),
 np.float64(-1.8279611373087934),
 np.float64(0.23428624702457235)]

In [94]:
1-res_min.fun

np.float64(0.903696114115064)

In [86]:
net = competition_network(network_data,network_exp,closeness_centrality_met=False,cutoff=0)
net.show("network_file.html")

Matching exp for 84.0% of interactions
network_file.html
